# WTA composition — formal stats

Four comparisons across full-hemisphere VOTC WTA composition.

| # | Comparison | Model | Per-category test |
|---|---|---|---|
| 1 | LH pt vs RH pt | `pct ~ category * intact_hemi + (1\|sid)` | Permutation |
| 2 | LH ctrl vs RH ctrl | `pct ~ category * hemi + (1\|sid)` | Permutation (paired) |
| 3 | LH ctrl vs LH pt | `pct ~ category * group + (1\|sid)` | Permutation |
| 4 | RH ctrl vs RH pt | `pct ~ category * group + (1\|sid)` | Permutation |

Primary inference: category × group/hemi interaction (LMM joint Wald χ², df=3).  
Per-category follow-ups: permutation test on mean difference (independent: label-shuffle; paired: sign-flip).  
Effect size: Cohen's d. Model fit: MSE (residual variance). Bonferroni α = .05/4 = .0125.

**Two analysis runs:**
- **Primary**: each control contributes both hemispheres. LMM `(1|sid)` handles non-independence. Standard published approach for OTC/lateralization work.
- **Supplementary**: resampled random ctrl-hemi allocations (1 hemi per ctrl). Provided per PI request to address independence concerns.

## Setup

In [16]:
import sys, os
from pathlib import Path
import numpy as np
import nibabel as nib
import pandas as pd
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/user_data/csimmon2/git_repos/sym_pt')
from sym_pt_params import (processed_dir, skip_subs, get_sessions,
                            is_patient, get_sub_info, _load_csv)

TFCE_DIR = Path(processed_dir) / 'group_results' / 'tfce_votc_fdr'
print(f'TFCE outputs: {TFCE_DIR}  exists: {TFCE_DIR.exists()}')

TFCE outputs: /user_data/csimmon2/sym_pt/group_results/tfce_votc_fdr  exists: True


## WTA computation

In [17]:
CATEGORIES = ['face', 'house', 'object', 'word']
COPES = {'face': 6, 'house': 7, 'object': 8, 'word': 9}
WTA_THRESHOLD = 2.326
EXTRA_SKIP = {'sub-017', 'control083', 'control085'}
PRE_SURGERY_SESSIONS = {
    'sub-021': {'01'}, 'sub-045': {'01'}, 'sub-047': {'01'}, 'sub-049': {'01'},
    'sub-070': {'01'}, 'sub-073': {'01'}, 'sub-081': {'01'}, 'sub-086': {'01'},
    'sub-108': {'02'},
}

VOTC_MASK_LH = nib.load(TFCE_DIR / 'votc_l_mask.nii.gz').get_fdata() > 0.5
VOTC_MASK_RH = nib.load(TFCE_DIR / 'votc_r_mask.nii.gz').get_fdata() > 0.5
print(f'LH VOTC: {VOTC_MASK_LH.sum():,} | RH VOTC: {VOTC_MASK_RH.sum():,}')

def load_subjects():
    df = _load_csv()
    subjects = {}
    for sc in sorted(df['sub_clean'].unique()):
        if sc in skip_subs: continue
        sid = f'sub-{sc}'
        sessions = get_sessions(sc)
        if not sessions or not os.path.exists(os.path.join(processed_dir, sid)):
            continue
        info = get_sub_info(sc, sessions[0])
        pt = is_patient(sc)
        intact = info.get('intact_hemi', '')
        group = info.get('group', 'unknown')
        if f'{group}{sc}' in EXTRA_SKIP or sid in EXTRA_SKIP or group == 'nonOTC':
            continue
        post = [f'{s:02d}' for s in sessions
                if f'{s:02d}' not in PRE_SURGERY_SESSIONS.get(sid, set())]
        if not post: continue
        subjects[sid] = {
            'session': post[0], 'group': group,
            'hemi': ('l' if intact == 'left' else 'r') if pt else None,
            'intact_hemi': intact,
        }
    return subjects

def _load_zstats(sid, ses):
    vols = {}
    for cat, cope in COPES.items():
        p = os.path.join(processed_dir, sid, f'ses-{ses}', 'derivatives', 'fsl',
                         'loc', 'HighLevel.gfeat', f'cope{cope}.feat', 'stats',
                         'zstat1_mni.nii.gz')
        if not os.path.exists(p): return None
        vols[cat] = nib.load(p).get_fdata()
    return vols

def _compute_wta(vols, mask):
    z = np.stack([vols[c][mask] for c in CATEGORIES], axis=-1)
    max_z = z.max(axis=-1)
    winner = z.argmax(axis=-1) + 1
    winner[max_z < WTA_THRESHOLD] = 0
    return winner

subjects = load_subjects()
wta_data, skipped = {}, []
for sid, info in subjects.items():
    vols = _load_zstats(sid, info['session'])
    if vols is None:
        skipped.append(sid); continue
    hemis = ['l', 'r'] if info['group'] == 'control' else [info['hemi']]
    for h in hemis:
        mask = VOTC_MASK_LH if h == 'l' else VOTC_MASK_RH
        wta_data[(sid, h)] = {
            'winner': _compute_wta(vols, mask),
            'group': info['group'],
            'intact_hemi': info.get('intact_hemi', 'both'),
        }
n_ctrl = sum(1 for k, v in wta_data.items() if v['group'] == 'control')
n_otc = sum(1 for k, v in wta_data.items() if v['group'] == 'OTC')
print(f'WTA computed: {n_ctrl} ctrl-hemis, {n_otc} OTC-hemis')
if skipped: print(f'Skipped (no zstats): {skipped}')

LH VOTC: 11,340 | RH VOTC: 11,540
WTA computed: 76 ctrl-hemis, 22 OTC-hemis
Skipped (no zstats): ['sub-098', 'sub-101', 'sub-109']


## Long-format dataframe builder

In [18]:
def build_wta_long(randomize_controls=False, seed=42):
    """randomize_controls=False (default): each ctrl contributes both hemis.
    =True: each ctrl contributes one random hemi (preserves independence)."""
    rng = np.random.default_rng(seed)
    ctrl_sids = sorted({sid for (sid, h), d in wta_data.items()
                        if d['group'] == 'control'})
    ctrl_choice = ({sid: rng.choice(['l', 'r']) for sid in ctrl_sids}
                   if randomize_controls else None)
    rows = []
    for (sid, h), d in wta_data.items():
        if d['group'] == 'control' and randomize_controls:
            if h != ctrl_choice[sid]: continue
        winner = d['winner']
        n_sel = (winner > 0).sum()
        if n_sel == 0: continue
        for i, cat in enumerate(CATEGORIES, start=1):
            rows.append({
                'sid': sid, 'hemi': h, 'group': d['group'],
                'intact_hemi': d.get('intact_hemi', 'both'),
                'category': cat,
                'pct': 100 * (winner == i).sum() / n_sel,
            })
    return pd.DataFrame(rows)

## Helpers — LMM, permutation, Cohen's d

In [19]:
ALPHA_BONF = 0.05 / 4
N_PERM = 10000

def fit_lmm(formula, data, groups):
    for method in ['powell', 'cg', 'bfgs', 'lbfgs']:
        try:
            fit = smf.mixedlm(formula, data, groups=data[groups]).fit(method=method)
            return fit, f'MixedLM ({method})'
        except Exception:
            continue
    return smf.ols(formula, data).fit(cov_type='HC3'), 'OLS+HC3 (LMM failed)'

def joint_wald(fit, term_substr):
    params = list(fit.params.index)
    target = [p for p in params if term_substr in p]
    if not target: return None, None, None
    L = np.zeros((len(target), len(params)))
    for i, p in enumerate(target):
        L[i, params.index(p)] = 1
    w = fit.wald_test(L, use_f=False, scalar=True)
    return float(w.statistic), float(w.pvalue), len(target)

def perm_independent(a, b, n_perm=N_PERM, seed=None):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    obs = a.mean() - b.mean()
    combined = np.concatenate([a, b])
    n_a = len(a)
    null = np.empty(n_perm)
    for i in range(n_perm):
        perm = rng.permutation(combined)
        null[i] = perm[:n_a].mean() - perm[n_a:].mean()
    p = ((np.abs(null) >= np.abs(obs)).sum() + 1) / (n_perm + 1)
    return float(obs), float(p)

def perm_paired(a, b, n_perm=N_PERM, seed=None):
    rng = np.random.default_rng(seed)
    diff = np.asarray(a) - np.asarray(b)
    obs = diff.mean()
    null = np.empty(n_perm)
    for i in range(n_perm):
        signs = rng.choice([-1, 1], size=len(diff))
        null[i] = (signs * diff).mean()
    p = ((np.abs(null) >= np.abs(obs)).sum() + 1) / (n_perm + 1)
    return float(obs), float(p)

def cohens_d_indep(a, b):
    a, b = np.asarray(a), np.asarray(b)
    pooled = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1))
                     / (len(a)+len(b)-2))
    return float((a.mean() - b.mean()) / pooled) if pooled > 0 else np.nan

def cohens_d_paired(a, b):
    diff = np.asarray(a) - np.asarray(b)
    sd = diff.std(ddof=1)
    return float(diff.mean() / sd) if sd > 0 else np.nan

## Analysis functions

Each returns dict with omnibus χ²/p, MSE, and per-category mean/diff/d/p (permutation).

In [20]:
def _percat_independent(df_a, df_b):
    out = {}
    for cat in CATEGORIES:
        a = df_a[df_a.category == cat]['pct'].values
        b = df_b[df_b.category == cat]['pct'].values
        diff, p = perm_independent(a, b)
        d = cohens_d_indep(a, b)
        out[cat] = {'m_a': float(a.mean()), 'm_b': float(b.mean()),
                    'diff': diff, 'p': p, 'd': d}
    return out

def _percat_paired(df):
    out = {}
    for cat in CATEGORIES:
        sub = df[df.category == cat]
        a = sub[sub.hemi == 'l'].set_index('sid')['pct']
        b = sub[sub.hemi == 'r'].set_index('sid')['pct']
        common = a.index.intersection(b.index)
        a_v, b_v = a.loc[common].values, b.loc[common].values
        diff, p = perm_paired(a_v, b_v)
        d = cohens_d_paired(a_v, b_v)
        out[cat] = {'m_a': float(a_v.mean()), 'm_b': float(b_v.mean()),
                    'diff': diff, 'p': p, 'd': d}
    return out

def _mse(fit):
    return float(fit.scale) if hasattr(fit, 'scale') else np.nan

def analysis_1(df_long):
    df = df_long[df_long.group == 'OTC'].copy()
    df['intact'] = df['hemi'].map({'l': 'LH', 'r': 'RH'})
    fit, ft = fit_lmm('pct ~ C(category) * C(intact)', df, 'sid')
    chi, p, dfw = joint_wald(fit, ':C(intact)')
    pc = _percat_independent(df[df.intact == 'LH'], df[df.intact == 'RH'])
    return {'label': 'LH pt vs RH pt', 'label_a': 'LH_pt', 'label_b': 'RH_pt',
            'chi': chi, 'p_omni': p, 'df_w': dfw, 'mse': _mse(fit),
            'percat': pc, 'paired': False}

def analysis_2(df_long):
    df = df_long[df_long.group == 'control'].copy()
    is_paired = (df.groupby('sid')['hemi'].nunique() > 1).any()
    fit, ft = fit_lmm('pct ~ C(category) * C(hemi)', df, 'sid')
    chi, p, dfw = joint_wald(fit, ':C(hemi)')
    pc = _percat_paired(df) if is_paired else _percat_independent(
        df[df.hemi == 'l'], df[df.hemi == 'r'])
    return {'label': 'LH ctrl vs RH ctrl', 'label_a': 'LH_ctrl', 'label_b': 'RH_ctrl',
            'chi': chi, 'p_omni': p, 'df_w': dfw, 'mse': _mse(fit),
            'percat': pc, 'paired': is_paired}

def analysis_3(df_long):
    df = df_long[df_long.hemi == 'l'].copy()
    fit, ft = fit_lmm('pct ~ C(category) * C(group)', df, 'sid')
    chi, p, dfw = joint_wald(fit, ':C(group)')
    pc = _percat_independent(df[df.group == 'control'], df[df.group == 'OTC'])
    return {'label': 'LH ctrl vs LH pt', 'label_a': 'ctrl', 'label_b': 'pt',
            'chi': chi, 'p_omni': p, 'df_w': dfw, 'mse': _mse(fit),
            'percat': pc, 'paired': False}

def analysis_4(df_long):
    df = df_long[df_long.hemi == 'r'].copy()
    fit, ft = fit_lmm('pct ~ C(category) * C(group)', df, 'sid')
    chi, p, dfw = joint_wald(fit, ':C(group)')
    pc = _percat_independent(df[df.group == 'control'], df[df.group == 'OTC'])
    return {'label': 'RH ctrl vs RH pt', 'label_a': 'ctrl', 'label_b': 'pt',
            'chi': chi, 'p_omni': p, 'df_w': dfw, 'mse': _mse(fit),
            'percat': pc, 'paired': False}

ANALYSES = [analysis_1, analysis_2, analysis_3, analysis_4]

## Reporter

In [21]:
def print_single(res, idx):
    print('=' * 70); print(f'Analysis {idx}: {res["label"]}'); print('=' * 70)
    if res['paired']: print('  Design: PAIRED (sign-flip perm)')
    sig = '*' if res['p_omni'] is not None and res['p_omni'] < .05 else ' '
    print(f'  Omnibus (LMM): χ²({res["df_w"]}) = {res["chi"]:.3f}, '
          f'p = {res["p_omni"]:.4f} {sig}    MSE = {res["mse"]:.2f}')
    print()
    print(f"  {'cat':<8} {res['label_a']:>8} {res['label_b']:>8} "
          f"{'diff':>7} {'d':>6} {'p_perm':>8}  bonf")
    for cat in CATEGORIES:
        c = res['percat'][cat]
        flag = '*' if c['p'] < ALPHA_BONF else ' '
        print(f"  {cat:<8} {c['m_a']:>8.2f} {c['m_b']:>8.2f} "
              f"{c['diff']:>+7.2f} {c['d']:>+6.2f} {c['p']:>8.4f}  {flag}")

## PRIMARY — non-independent controls (LMM handles via `(1|sid)`)

In [22]:
wta_long = build_wta_long(randomize_controls=False)
n_ctrl = wta_long[wta_long.group == 'control'].sid.nunique()
n_otc = wta_long[wta_long.group == 'OTC'].sid.nunique()
print(f'Primary: {n_ctrl} ctrl (both hemis) + {n_otc} OTC\n')

primary_results = {}
for idx, fn in enumerate(ANALYSES, start=1):
    res = fn(wta_long)
    primary_results[idx] = res
    print_single(res, idx); print()

Primary: 38 ctrl (both hemis) + 22 OTC

Analysis 1: LH pt vs RH pt
  Omnibus (LMM): χ²(3) = 3.757, p = 0.2889      MSE = 433.49

  cat         LH_pt    RH_pt    diff      d   p_perm  bonf
  face        23.70    23.98   -0.28  -0.01   0.9768   
  house       33.12    19.05  +14.07  +0.95   0.0383   
  object      22.45    28.14   -5.70  -0.29   0.5441   
  word        20.73    28.83   -8.09  -0.31   0.4846   

Analysis 2: LH ctrl vs RH ctrl
  Design: PAIRED (sign-flip perm)
  Omnibus (LMM): χ²(3) = 32.791, p = 0.0000 *    MSE = 141.63

  cat       LH_ctrl  RH_ctrl    diff      d   p_perm  bonf
  face        20.20    13.06   +7.14  +0.87   0.0001  *
  house       35.34    27.58   +7.76  +1.10   0.0001  *
  object      34.79    45.58  -10.79  -1.04   0.0001  *
  word         9.68    13.79   -4.11  -0.57   0.0007  *

Analysis 3: LH ctrl vs LH pt
  Omnibus (LMM): χ²(3) = 11.023, p = 0.0116 *    MSE = 225.71

  cat          ctrl       pt    diff      d   p_perm  bonf
  face        20.20    2

## SUPPLEMENTARY — resampled random ctrl-hemi allocation

Provided per PI request. Each iteration randomly assigns each control to one hemisphere (preserving independence). Reports median across iterations.

In [ ]:
N_ITER = 500
supp_results = {i: [] for i in range(1, 5)}
for it in range(N_ITER):
    df = build_wta_long(randomize_controls=True, seed=42 + it)
    for idx, fn in enumerate(ANALYSES, start=1):
        supp_results[idx].append(fn(df))
    if (it + 1) % 50 == 0:
        print(f'  iter {it+1}/{N_ITER}')
print('Done.\n')

for idx in range(1, 5):
    runs = supp_results[idx]
    label = runs[0]['label']
    label_a, label_b = runs[0]['label_a'], runs[0]['label_b']
    print('=' * 78); print(f'Analysis {idx}: {label}   (n_iter = {len(runs)})')
    print('=' * 78)
    chis = np.array([r['chi'] for r in runs])
    p_omnis = np.array([r['p_omni'] for r in runs])
    mses = np.array([r['mse'] for r in runs])
    print(f'  Omnibus (LMM): median χ²({runs[0]["df_w"]}) = {np.median(chis):.2f}  '
          f'median p = {np.median(p_omnis):.4f}   median MSE = {np.median(mses):.2f}')
    print()
    print(f"  {'cat':<8} {label_a+'_med':>10} {label_b+'_med':>10} "
          f"{'med_diff':>9} {'med_d':>7} {'med_p':>8}")
    for cat in CATEGORIES:
        m_a = [r['percat'][cat]['m_a'] for r in runs]
        m_b = [r['percat'][cat]['m_b'] for r in runs]
        diffs = [r['percat'][cat]['diff'] for r in runs]
        ds = [r['percat'][cat]['d'] for r in runs]
        ps = [r['percat'][cat]['p'] for r in runs]
        print(f"  {cat:<8} {np.median(m_a):>10.2f} {np.median(m_b):>10.2f} "
              f"{np.median(diffs):>+9.2f} {np.median(ds):>+7.2f} {np.median(ps):>8.4f}")
    print()

## Summary — all results in one place

Pulls from `primary_results` and `supp_results`. Compact tables for write-up.

In [ ]:
print('=' * 78)
print('OMNIBUS LMM (category × group/hemi interaction, joint Wald χ² df=3)')
print('=' * 78)
print(f"  {'Analysis':<22} {'PRIMARY':>22}    {'SUPPLEMENTARY (median)':>25}")
print(f"  {'':<22} {'χ²':>7} {'p':>8} {'MSE':>6}    "
      f"{'χ²':>7} {'p':>8} {'MSE':>6}")
for idx in range(1, 5):
    p = primary_results[idx]
    s = supp_results[idx]
    s_chi = np.median([r['chi'] for r in s])
    s_p = np.median([r['p_omni'] for r in s])
    s_mse = np.median([r['mse'] for r in s])
    p_sig = '*' if p['p_omni'] < .05 else ' '
    s_sig = '*' if s_p < .05 else ' '
    print(f"  {p['label']:<22} {p['chi']:>7.2f} {p['p_omni']:>7.4f}{p_sig} {p['mse']:>6.1f}    "
          f"{s_chi:>7.2f} {s_p:>7.4f}{s_sig} {s_mse:>6.1f}")

print('\n' + '=' * 78)
print('PER-CATEGORY (permutation test on mean difference)')
print('=' * 78)
for idx in range(1, 5):
    p = primary_results[idx]
    s = supp_results[idx]
    label_a, label_b = p['label_a'], p['label_b']
    print(f"\n  Analysis {idx}: {p['label']}")
    print(f"  {'':<8} {'PRIMARY':>26}      {'SUPPLEMENTARY (med)':>26}")
    print(f"  {'cat':<8} {'diff':>7} {'d':>6} {'p':>8}      "
          f"{'diff':>7} {'d':>6} {'p':>8}")
    for cat in CATEGORIES:
        pc_p = p['percat'][cat]
        diffs_s = [r['percat'][cat]['diff'] for r in s]
        ds_s = [r['percat'][cat]['d'] for r in s]
        ps_s = [r['percat'][cat]['p'] for r in s]
        p_flag = '*' if pc_p['p'] < ALPHA_BONF else ("'" if pc_p['p'] < .05 else ' ')
        s_p_med = np.median(ps_s)
        s_flag = '*' if s_p_med < ALPHA_BONF else ("'" if s_p_med < .05 else ' ')
        print(f"  {cat:<8} {pc_p['diff']:>+7.2f} {pc_p['d']:>+6.2f} "
              f"{pc_p['p']:>7.4f}{p_flag}      "
              f"{np.median(diffs_s):>+7.2f} {np.median(ds_s):>+6.2f} "
              f"{s_p_med:>7.4f}{s_flag}")

print("\n  * = p < .0125 (Bonferroni)   ' = p < .05 uncorrected")

print('\n' + '=' * 78)
print('DIRECTIONAL PATTERN (vs control baseline)')
print('=' * 78)
print(f"  {'Category':<10} {'Predicted':>10}    {'LH pt (A3)':>22}    {'RH pt (A4)':>22}")
for cat, pred in [('face', '↑'), ('house', '↓'), ('object', '↓'), ('word', '↑')]:
    a3 = primary_results[3]['percat'][cat]
    a4 = primary_results[4]['percat'][cat]
    # diff = ctrl - pt, so pt direction is OPPOSITE sign of diff
    a3_dir = '↓' if a3['diff'] > 0 else '↑'
    a4_dir = '↓' if a4['diff'] > 0 else '↑'
    a3_match = '✓' if a3_dir == pred else '✗'
    a4_match = '✓' if a4_dir == pred else '✗'
    a3_sig = ('***' if a3['p'] < ALPHA_BONF else '*' if a3['p'] < .05 else 'ns')
    a4_sig = ('***' if a4['p'] < ALPHA_BONF else '*' if a4['p'] < .05 else 'ns')
    print(f"  {cat:<10} {pred:>10}    {a3_dir} {a3_match}  d={a3['d']:>+.2f}  {a3_sig:>5}    "
          f"{a4_dir} {a4_match}  d={a4['d']:>+.2f}  {a4_sig:>5}")
print("\n  *** = p < .0125 (Bonferroni)   * = p < .05 uncorrected   ✓/✗ = direction match")

OMNIBUS LMM (category × group/hemi interaction, joint Wald χ² df=3)
  Analysis                              PRIMARY       SUPPLEMENTARY (median)
                              χ²        p    MSE         χ²        p    MSE
  LH pt vs RH pt            3.76  0.2889   433.5       3.76  0.2889   433.5
  LH ctrl vs RH ctrl       32.79  0.0000*  141.6      17.50  0.0006*  141.9
  LH ctrl vs LH pt         11.02  0.0116*  225.7       7.90  0.0481*  266.1
  RH ctrl vs RH pt         33.89  0.0000*  181.7      22.27  0.0001*  226.4

PER-CATEGORY (permutation test on mean difference)

  Analysis 1: LH pt vs RH pt
                              PRIMARY             SUPPLEMENTARY (med)
  cat         diff      d        p         diff      d        p
  face       -0.28  -0.01  0.9765         -0.28  -0.01  0.9745 
  house     +14.07  +0.95  0.0426'       +14.07  +0.95  0.0392'
  object     -5.70  -0.29  0.5425         -5.70  -0.29  0.5438 
  word       -8.09  -0.31  0.4846         -8.09  -0.31  0.4837 

  